# Lab Week 4 · Evaluate and Explain
Guided practice for clustering evaluation. Run cells in order with **Shift + Enter**. Reference labels are used for evaluation only, never for clustering or scaling. Explain decisions in the supplied cells. Save your notebook and export its current outputs as PDF.

This practice notebook is separate from **Assessed Lab Exercise 1 (5%)**. Your lecturer supplies the assessed brief, dataset, deadline and submission requirements separately.

In [ ]:
# First startup downloads libraries into the browser's Python environment.
import piplite
await piplite.install(["numpy", "pandas", "matplotlib", "scikit-learn"])
print("Libraries ready. Continue with Task 1.")

## Task 1 · Inspect and prepare
Use a synthetic customer example: monthly visits and spending. Generator labels are only a reference partition, not verified customer types. Check scales before using Euclidean distances.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (silhouette_score, silhouette_samples,
    davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score)
from sklearn.metrics.cluster import contingency_matrix

raw, reference = make_blobs(n_samples=150,
    centers=[[2, 1800], [8, 2200], [5, 8000]],
    cluster_std=[[0.6, 600], [1.0, 800], [1.0, 1100]], random_state=41)
df = pd.DataFrame(raw, columns=["monthly_visits", "monthly_spending_MUR"])
print(df.head())
print(df.describe().round(2))
print("Missing values:", df.isna().sum().to_dict())
scaler = StandardScaler()
X = scaler.fit_transform(df)  # Never include reference labels as features.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df.iloc[:, 0], df.iloc[:, 1], color="teal", s=20)
axes[0].set(title="Original measurements", xlabel="Monthly visits", ylabel="Spending (MUR)")
axes[1].scatter(X[:, 0], X[:, 1], color="teal", s=20)
axes[1].set(title="Standardised features", xlabel="Visits (standardised)", ylabel="Spending (standardised)")
axes[1].set_aspect("equal", adjustable="datalim")
plt.tight_layout()
plt.show()

### Your explanation
Why could spending dominate an unscaled Euclidean distance? Explain why the reference labels must stay outside X.

In [ ]:
# Explain your choices and interpret the evidence here.
# Your answer:

## Task 2 · Fit three methods
Use exactly the same prepared observations for every fit. Cluster IDs are arbitrary. DBSCAN may leave observations unassigned as noise (−1).

In [ ]:
def make_model(name, seed=41, n_init=12):
    if name == "K-means":
        return KMeans(n_clusters=3, n_init=n_init, random_state=seed)
    if name == "Agglomerative":
        return AgglomerativeClustering(n_clusters=3, linkage="ward")
    return DBSCAN(eps=0.45, min_samples=5)

models = {name: make_model(name).fit(X)
          for name in ["K-means", "Agglomerative", "DBSCAN"]}
labels = {name: model.labels_ for name, model in models.items()}

def plot_partition(ax, values, assigned, title):
    assigned = np.asarray(assigned)
    keep = assigned >= 0
    ax.scatter(values[keep, 0], values[keep, 1], c=assigned[keep], cmap="tab10", s=22)
    if (~keep).any():
        ax.scatter(values[~keep, 0], values[~keep, 1], c="grey", marker="x", label="Noise")
        ax.legend()
    ax.set(title=title, xlabel="Standardised feature 1", ylabel="Standardised feature 2")
    ax.set_aspect("equal", adjustable="datalim")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, assigned) in zip(axes, labels.items()):
    plot_partition(ax, X, assigned, name)
plt.tight_layout()
plt.show()

### Your explanation
Which observations are assigned differently? Explain each method’s grouping assumption and the meaning of DBSCAN noise.

In [ ]:
# Explain your choices and interpret the evidence here.
# Your answer:

## Task 3 · Evaluate from within
Exclude noise from internal scores and report coverage. A metric requires at least two groups and fewer groups than retained observations. Scores favour particular geometry; no score proves practical usefulness. Subset scores at different DBSCAN coverage levels are not directly comparable.

In [ ]:
def internal_report(values, assigned):
    assigned = np.asarray(assigned)
    keep = assigned >= 0
    scored = values[keep]
    groups = assigned[keep]
    k = len(np.unique(groups))
    row = {"clusters": k, "retained": int(keep.sum()),
           "noise": int((~keep).sum()), "total": len(values)}
    if 2 <= k < len(scored):
        row.update(silhouette=silhouette_score(scored, groups),
                   davies_bouldin=davies_bouldin_score(scored, groups),
                   calinski_harabasz=calinski_harabasz_score(scored, groups))
    else:
        row.update(silhouette=np.nan, davies_bouldin=np.nan,
                   calinski_harabasz=np.nan)
    return row

report = pd.DataFrame({name: internal_report(X, assigned)
                       for name, assigned in labels.items()}).T
print(report.round(3))

# Inspect the silhouette distribution; a mean can hide weak assignments.
sample_scores = silhouette_samples(X, labels["K-means"])
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(np.arange(len(X)), np.sort(sample_scores), color="teal", width=1)
ax.axhline(sample_scores.mean(), color="darkorange", linestyle="--", label="Mean")
ax.set(xlabel="Observations sorted by silhouette", ylabel="Silhouette", ylim=(-1, 1))
ax.legend()
plt.tight_layout()
plt.show()

# A one-cluster result is deliberately invalid for these scores.
print("One-cluster check:", internal_report(X, np.zeros(len(X), dtype=int)))

### Your explanation
Interpret all three metrics and their preferred direction. Does their ranking agree with the plots? Why must you report coverage and inspect individual silhouettes?

In [ ]:
# Explain your choices and interpret the evidence here.
# Your answer:

## Task 4 · Compare with reference labels
Only now use the generator labels for evaluation. ARI is adjusted for chance; NMI with arithmetic normalization is not. External evaluation uses every observation, treating noise −1 as one category. This convention does not make noise an ordinary discovered cluster.

In [ ]:
external = pd.DataFrame({name: {
    "ARI": adjusted_rand_score(reference, assigned),
    "NMI": normalized_mutual_info_score(reference, assigned, average_method="arithmetic")}
    for name, assigned in labels.items()}).T
print(external.round(3))
assigned = labels["K-means"]
print("Reference groups x fitted clusters:")
print(contingency_matrix(reference, assigned))
# A one-to-one renaming preserves the partition.
renamed = np.array([100 - group if group >= 0 else -1 for group in assigned])
print("Original ARI:", adjusted_rand_score(reference, assigned))
print("Renamed ARI:", adjusted_rand_score(reference, renamed))
print("Original NMI:", normalized_mutual_info_score(reference, assigned))
print("Renamed NMI:", normalized_mutual_info_score(reference, renamed))

### Your explanation
Explain label-renaming invariance. Why might reference agreement and internal quality disagree? Would a reference category always match a useful business segment?

In [ ]:
# Explain your choices and interpret the evidence here.
# Your answer:

## Task 5 · Sensitivity to noise and outliers
Perturb the prepared features and append unusual observations. Retain original IDs for comparisons. Keep the original scaler fixed: this experiment changes observations, rather than simultaneously changing the feature transformation.

In [ ]:
rng = np.random.default_rng(71)
perturbed = X + rng.normal(0, 0.12, size=X.shape)
added = rng.uniform(-4, 4, size=(10, X.shape[1]))
changed_X = np.vstack([perturbed, added])
changed_labels = {name: make_model(name).fit_predict(changed_X) for name in models}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, assigned) in zip(axes, changed_labels.items()):
    plot_partition(ax, changed_X, assigned, name + " with changes")
plt.tight_layout()
plt.show()
print(pd.DataFrame({name: internal_report(changed_X, assigned)
                    for name, assigned in changed_labels.items()}).T.round(3))
for name, assigned in changed_labels.items():
    # Compare only matching original observations, not the appended outliers.
    print(name, "original-ID ARI:", round(adjusted_rand_score(labels[name], assigned[:len(X)]), 3))

### Your explanation
What changed in boundaries, noise and internal scores? Why is added-outlier agreement calculated on original IDs only? Investigate unusual observations rather than automatically deleting them.

In [ ]:
# Explain your choices and interpret the evidence here.
# Your answer:

## Task 6 · Stability and interpretation
Repeat perturbations. K-means uses a single start with a different seed per trial to expose initialisation sensitivity as well. High stability means agreement under this experiment—not proof of a correct or useful partition.

In [ ]:
records = []
for seed in range(10):
    trial_rng = np.random.default_rng(100 + seed)
    trial_X = np.vstack([X + trial_rng.normal(0, 0.12, X.shape),
                         trial_rng.uniform(-4, 4, size=(10, X.shape[1]))])
    for name in models:
        trial_labels = make_model(name, seed=seed, n_init=1).fit_predict(trial_X)
        records.append({"trial": seed + 1, "method": name,
            "ARI": adjusted_rand_score(labels[name], trial_labels[:len(X)]),
            "noise": int((trial_labels == -1).sum())})
stability = pd.DataFrame(records)
print(stability.groupby("method")["ARI"].agg(["mean", "min", "max"]).round(3))
fig, ax = plt.subplots(figsize=(8, 4))
for name in models:
    rows = stability[stability.method == name]
    ax.plot(rows.trial, rows.ARI, marker="o", label=name)
ax.set(xlabel="Trial", ylabel="ARI vs original partition", ylim=(-1, 1.05))
ax.legend()
plt.tight_layout()
plt.show()

# Profile observed measurements, not the standardised coordinates.
profiles = df.assign(cluster=labels["K-means"]).groupby("cluster").agg(
    size=("monthly_visits", "size"),
    mean_visits=("monthly_visits", "mean"),
    mean_spending=("monthly_spending_MUR", "mean"))
print(profiles.round(2))

### Your explanation
Summarise stability and propose descriptions for the segments. Suggest one possible application and one missing piece of context. Why could an all-noise result be stable but useless?

In [ ]:
# Explain your choices and interpret the evidence here.
# Your answer:

## Task 7 · Your turn: independent analysis
The new dataset below supplies measurements only; no worked solution follows.

1. Inspect scales and justify preprocessing.
2. Compare at least two clustering methods and several plausible parameter settings on the same prepared data.
3. Report appropriate internal scores, retained counts and noise. Explain invalid cases.
4. Test sensitivity and stability on matching observation IDs.
5. Describe groups using feature summaries and plots.
6. Recommend an approach with evidence and limitations. Reference labels are unavailable here: explain what this prevents you from claiming.

This independent practice is not the separate assessed-lab brief.

In [ ]:
practice_rng = np.random.default_rng(87)
a = practice_rng.normal([3, 1500], [0.5, 500], size=(55, 2))
b = practice_rng.normal([7, 6500], [1.5, 1400], size=(60, 2))
unusual = practice_rng.uniform([1, 500], [12, 10000], size=(10, 2))
practice_df = pd.DataFrame(np.vstack([a, b, unusual]),
    columns=["monthly_visits", "monthly_spending_MUR"])
practice_df.index = pd.Index(np.arange(1, len(practice_df) + 1), name="observation_id")
print(practice_df.head())
print("Observations:", len(practice_df))
# IDs identify observations; they are not clustering features.

### Your work · Inspect and prepare

In [ ]:
# Add your code, plots and explanation here.

### Your work · Fit and evaluate

In [ ]:
# Add your code, plots and explanation here.

### Your work · Robustness and stability

In [ ]:
# Add your code, plots and explanation here.

### Your work · Profiles and justified recommendation

In [ ]:
# Add your code, plots and explanation here.